# Go1 on rough terrain: feasibility spike A2 (training) and exports for A3

Part of the lab's applied quadruped (`docs/proposals/mujoco-quadruped.md`). Criteria fixed before measuring in
`openspec/changes/add-applied-mujoco-spike/design.md`:

- **A2 (training).** MuJoCo Playground's default PPO configuration for `Go1JoystickRoughTerrain` trains within
  **60 minutes** of wall-clock time on the Colab GPU, and the policy walks: in 10 episodes of 20 s with a forward
  command of 0.5 m/s, **at most 1 fall** and a **mean forward speed ≥ 0.35 m/s**.
- The notebook also exports the policy (normalizer and MLP) and 100 observation–action pairs for **A3**, the
  TypeScript equivalence test and the browser run.

**Before running:** Runtime → Change runtime type → GPU (an A100 or L4 if available; the GPU model is recorded).
Run all cells; the last one downloads `go1-a2-results.zip`. Put its contents in `applied/mujoco-quadruped/results/`.

In [ ]:
# JAX is pinned to 0.9.2: brax 0.14.2 (the latest) calls jax.device_put_replicated, removed in JAX 0.10.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q playground==0.2.0 mujoco==3.14.0 mujoco-mjx==3.14.0 jax[cuda12]==0.9.2 jaxlib==0.9.2 brax==0.14.2 warp-lang==1.17.0

In [ ]:
import functools, json, subprocess, time
import jax, jax.numpy as jp
import numpy as np
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from flax import serialization
from mujoco_playground import registry, wrapper
from mujoco_playground.config import locomotion_params

ENV = 'Go1JoystickRoughTerrain'
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', GPU, '| devices:', jax.devices())
env_cfg = registry.get_default_config(ENV)
env = registry.load(ENV, config=env_cfg)
ppo_params = locomotion_params.brax_ppo_config(ENV)
print(ppo_params)

## A2: training (Playground's default PPO configuration, unchanged)

In [ ]:
progress = []
t_start = time.time()
def progress_fn(steps, metrics):
    progress.append({'steps': int(steps), 'minutes': (time.time() - t_start) / 60, 'reward': float(metrics.get('eval/episode_reward', np.nan))})
    print(f"{steps:>12,} steps  {progress[-1]['minutes']:6.1f} min  eval reward {progress[-1]['reward']:.2f}")

params_cfg = dict(ppo_params)
network_factory = functools.partial(ppo_networks.make_ppo_networks, **params_cfg.pop('network_factory'))
train_fn = functools.partial(ppo.train, **params_cfg, network_factory=network_factory, progress_fn=progress_fn)
make_inference_fn, params, _ = train_fn(environment=env, eval_env=registry.load(ENV, config=env_cfg), wrap_env_fn=wrapper.wrap_for_brax_training)
train_minutes = (time.time() - t_start) / 60
print(f'training: {train_minutes:.1f} min (JIT compilation included)')

## A2: does it walk? 10 episodes of 20 s, forward command 0.5 m/s, rough terrain, no observation noise

In [ ]:
eval_cfg = registry.get_default_config(ENV)
eval_cfg.noise_config.level = 0.0
eval_env = registry.load(ENV, config=eval_cfg)
reset, step = jax.jit(eval_env.reset), jax.jit(eval_env.step)
policy = jax.jit(make_inference_fn(params, deterministic=True))
COMMAND = jp.array([0.5, 0.0, 0.0])
STEPS = int(round(20.0 / eval_env.dt))
episodes = []
pairs = []  # (observation, action) pairs for the A3 equivalence test
for ep in range(10):
    state = reset(jax.random.PRNGKey(1000 + ep))
    state.info['command'] = COMMAND
    # The reset draws a random heading: progress is measured along the robot's initial forward direction.
    w, x, y, z = (float(v) for v in state.data.qpos[3:7])
    heading = np.array([1 - 2 * (y * y + z * z), 2 * (x * y + w * z)])  # body x axis in the world xy plane
    heading /= np.linalg.norm(heading)
    p0 = np.asarray(state.data.qpos[0:2], dtype=float)
    fell, t = False, 0
    for t in range(STEPS):
        state.info['command'] = COMMAND
        action, _ = policy(state.obs, jax.random.PRNGKey(0))
        if ep == 0 and t % 10 == 0 and len(pairs) < 100:
            pairs.append({'state': np.asarray(state.obs['state']).tolist(), 'action': np.asarray(action).tolist()})
        state = step(state, action)
        if float(state.done) > 0:
            fell = True
            break
    seconds = (t + 1) * eval_env.dt
    forward = float(np.dot(np.asarray(state.data.qpos[0:2], dtype=float) - p0, heading))
    episodes.append({'seed': 1000 + ep, 'fell': fell, 'seconds': seconds, 'forward_m': forward})
    print(episodes[-1])
falls = sum(e['fell'] for e in episodes)
speed = float(np.mean([e['forward_m'] / 20.0 for e in episodes]))
a2 = {'train_minutes': train_minutes, 'falls': falls, 'mean_forward_speed': speed,
      'pass': bool(train_minutes <= 60 and falls <= 1 and speed >= 0.35)}
print('A2', 'PASS' if a2['pass'] else 'FAIL', a2)

## Exports for A3: policy parameters, environment constants, observation–action pairs

In [ ]:
def to_lists(tree):
    return jax.tree_util.tree_map(lambda x: np.asarray(x).tolist(), serialization.to_state_dict(tree))

export = {
    'env': ENV,
    'playground': '0.2.0',
    'gpu': GPU,
    'normalizer': to_lists(params[0]),
    'policy': to_lists(params[1]),
    'network': {'hidden': list(ppo_params.network_factory.policy_hidden_layer_sizes), 'activation': 'swish', 'output': 'tanh(loc) of a normal-tanh distribution'},
    'constants': {
        'ctrl_dt': float(eval_env.dt), 'sim_dt': float(eval_env.sim_dt), 'action_scale': float(env_cfg.action_scale),
        'Kp': float(env_cfg.Kp), 'Kd': float(env_cfg.Kd), 'default_pose': np.asarray(eval_env._default_pose).tolist(),
        'command': COMMAND.tolist(),
    },
    'pairs': pairs,
}
results = {'gpu': GPU, 'ppo': {k: (v if not hasattr(v, 'to_dict') else v.to_dict()) for k, v in dict(ppo_params).items() if k != 'network_factory'},
           'progress': progress, 'episodes': episodes, 'a2': a2}
json.dump(export, open('go1-policy.json', 'w'))
json.dump(results, open('go1-a2-results.json', 'w'), indent=2)
!zip -q go1-a2-results.zip go1-policy.json go1-a2-results.json
from google.colab import files
files.download('go1-a2-results.zip')